# DeltaGrad Colab bootstrap

Clones this repo fresh from GitHub every session onto Colab's local disk (git
operations over the Drive FUSE mount are slow, so the working tree lives in
`/content`, not Drive) runs an experiment, then commits + pushes the results
straight back to GitHub. Drive is only mounted to cache the CIFAR-100 tarball
across sessions (~170MB, otherwise re-downloaded on every fresh runtime).

**One-time setup (before your first run):**
1. On your local machine, push this repo to GitHub if you haven't yet:
   `git remote add origin <url>` then `git push -u origin master`.
2. Generate a GitHub Personal Access Token scoped to this repo with **Contents:
   read and write** permission: https://github.com/settings/tokens (fine-grained
   token, repository access limited to this repo only).
3. In Colab's left sidebar, click the key icon (Secrets) -> add a new secret
   named `GITHUB_TOKEN` -> paste the token -> enable "Notebook access".
4. Fill in `GITHUB_REPO`, `GIT_USER_NAME`, `GIT_USER_EMAIL` in the cell below.

After that, **Runtime > Run all** pulls the latest code, runs the experiment,
and pushes the results back -- no manual file copying either direction.

**Security note:** the token is embedded in the git remote URL for this session
only. Never `print()` `REMOTE_URL` or run `!git remote -v` -- either would leak
the token in plaintext into a cell output, which persists if you save/share the
notebook.

In [ ]:
from google.colab import drive, userdata
import os

# --- fill these in once ---
GITHUB_REPO = "xandasoneill/deltagrad_optimizer"   # e.g. "xandasoneill/deltagrad"
GIT_USER_NAME = "xandasoneill"
GIT_USER_EMAIL = "xandas.oneill@gmail.com"
# ---------------------------

drive.mount('/content/drive')
DATA_CACHE = '/content/drive/MyDrive/deltagrad_data_cache'
os.makedirs(DATA_CACHE, exist_ok=True)

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_ROOT = '/content/deltagrad_optimizer'
REMOTE_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Git sync: clone fresh, or pull latest if this runtime already has a clone
# (e.g. you're re-running this cell later in the same session).
if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    !git clone {REMOTE_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull origin master

!cd {REPO_ROOT} && git config user.email "{GIT_USER_EMAIL}" && git config user.name "{GIT_USER_NAME}"

import sys
sys.path.append(REPO_ROOT)
!pip install -q -r {REPO_ROOT}/requirements.txt

In [ ]:
# Stage the CIFAR-100 tarball from the Drive cache onto local (fast) disk, if
# we've cached it from a previous session -- otherwise torchvision just
# downloads it fresh below, and we cache it to Drive afterwards for next time.
os.makedirs(f'{REPO_ROOT}/data', exist_ok=True)
cached_tarball = f'{DATA_CACHE}/cifar-100-python.tar.gz'

if os.path.exists(cached_tarball):
    !cp {cached_tarball} {REPO_ROOT}/data/
else:
    print("No cached tarball yet -- torchvision will download it fresh (~170MB).")

In [ ]:
os.chdir(REPO_ROOT)

# Full-scale run of a core benchmark task against one optimizer, per
# deltagradpaperplan.pdf Table 1. See experiments/configs.py::TASK_REGISTRY for
# every --task choice, and --optimizer for windowed/ema/adam/adamw/
# sgd_momentum/adagrad/rmsprop.
!python -m experiments.run_task --task cifar100_noise_20 --optimizer windowed

# Cache the tarball to Drive for next session, if it wasn't already cached.
if not os.path.exists(cached_tarball):
    !cp {REPO_ROOT}/data/cifar-100-python.tar.gz {DATA_CACHE}/

In [ ]:
# Commit + push this run's results straight back to GitHub. The `git diff
# --cached --quiet ||` guard skips the commit (without erroring) if a rerun
# produced byte-identical results and there's nothing new to commit.
import time

os.chdir(REPO_ROOT)
!git add results/
commit_msg = f"Colab run: {time.strftime('%Y-%m-%d %H:%M:%S')}"
!git diff --cached --quiet || git commit -m "{commit_msg}"
!git push origin master